<a href="https://colab.research.google.com/github/tazir-shaif/ai-engineering-portfolio/blob/main/module-5-agents/Module_5_Session_4_Agent_Memory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 5 — Session 4: Agent Memory

## What are we building?
A Swiggy support agent with two types of memory:
1. Short term — remembers the full current conversation
2. Long term — remembers customer history across sessions

## Why does this matter?
A memoryless agent is frustrating to use. Real production agents
at Swiggy, Zepto, and Flipkart remember who you are and what
happened before. Memory is what makes an agent feel intelligent.

## What we will learn:
- How short term memory works using conversation history
- How long term memory works using a simple key-value store
- How to personalise agent responses using customer history
- Why memory management is critical at scale

## Model: openai/gpt-oss-20b on Groq
## No new libraries this session
## Platform: Google Colab

# Module 5 — Session 4: Agent Memory

## What are we building?
A Swiggy support agent with two types of real memory:
1. Short term — LangGraph SQLite checkpointer remembers
   the full current conversation across multiple turns
2. Long term — Mem0 remembers customer history across sessions

## Why does this matter?
A memoryless agent is frustrating. Real production agents at
Swiggy, Zepto, and Flipkart remember who you are and what
happened before. Memory is what makes an agent feel intelligent.

## What we will learn:
- Short term memory with LangGraph SQLite checkpointer
- Long term memory with Mem0 (industry standard, 47K GitHub stars)
- How to personalise agent responses using customer history
- How both memory types work together in one agent

## Tools:
- LangGraph SQLite checkpointer — short term memory (free, no account)
- Mem0 free tier — long term memory (free, no credit card)

## Model: openai/gpt-oss-20b on Groq
## New library: mem0ai
## Platform: Google Colab

## Step 1: Install and Setup
mem0ai is our one new library this session.
langgraph and langchain-groq we already know from Session 3.

In [ ]:
# mem0ai — new library this session for long term memory
# langgraph-checkpoint-sqlite — separate package needed for SQLite checkpointer
!pip install mem0ai langgraph langchain-groq langchain langgraph-checkpoint-sqlite -q

# LangGraph imports
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.sqlite import SqliteSaver  # saves conversation state to SQLite file
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage, AIMessage
from langchain_core.tools import tool

# Mem0 import
from mem0 import MemoryClient                        # long term memory client

# Standard imports
from typing import TypedDict, Annotated
import operator
import os
import json
from google.colab import userdata

# Setup API keys
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
os.environ["MEM0_API_KEY"] = userdata.get("MEM0_API_KEY")

# Groq LLM
llm = ChatGroq(model="openai/gpt-oss-20b")

# Mem0 client — connects to Mem0 cloud using your API key
mem0 = MemoryClient()

print("Setup complete")
print("LLM:", llm.model_name)
print("Mem0 client ready:", type(mem0).__name__)

## Step 2: Define Tools
Same three Swiggy tools as before.
No changes here — tools are not affected by memory.
Memory is added at the agent level, not the tool level.

In [ ]:
@tool
def check_order_status(order_id: str) -> str:
    """Check the current status of a Swiggy order using the order ID.
    Use this first before deciding on refunds or any other action."""
    orders = {
        "ORD001": {"status": "Delayed", "eta": "45 minutes", "restaurant": "Burger King"},
        "ORD002": {"status": "Delivered", "eta": None, "restaurant": "Pizza Hut"},
        "ORD003": {"status": "Preparing", "eta": "25 minutes", "restaurant": "McDonald's"},
    }
    order = orders.get(order_id)
    if order:
        return f"Order {order_id} from {order['restaurant']}: {order['status']}. ETA: {order['eta']}"
    else:
        return f"Order {order_id} not found in system."

@tool
def calculate_refund(order_id: str, reason: str) -> str:
    """Calculate refund for a Swiggy order.
    Use this only after checking order status.
    Reason must be either wrong_item or late_delivery."""
    order_values = {"ORD001": 450, "ORD002": 320, "ORD003": 180}
    amount = order_values.get(order_id, 0)
    if reason == "wrong_item":
        refund = amount * 1.0
    elif reason == "late_delivery":
        refund = amount * 0.3
    else:
        refund = 0
    return f"Refund for {order_id}: ₹{refund} approved for reason: {reason}"

@tool
def escalate_to_human(order_id: str, reason: str) -> str:
    """Escalate the issue to a human support agent.
    Use this when you cannot resolve the issue with available tools."""
    return f"Order {order_id} escalated to human support. Reason: {reason}. Ticket created."

# Bind tools to LLM
tools = [check_order_status, calculate_refund, escalate_to_human]
llm_with_tools = llm.bind_tools(tools)

# Tool executor
tool_executor = {t.name: t for t in tools}

print("Tools ready:", [t.name for t in tools])

## Step 3: Short Term Memory with SQLite Checkpointer
Short term memory = remembering the current conversation across multiple turns.
LangGraph's SqliteSaver saves the full conversation state to a SQLite file
after every step. When the customer sends a follow up message, the agent
reads the saved state and continues from where it left off.
This replaces the manual messages list we managed ourselves in Session 2.

In [ ]:
# Define agent state — same as Session 3
class AgentState(TypedDict):
    messages: Annotated[list, operator.add]

# --- NODE 1: Agent Node ---
def agent_node(state: AgentState):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

# --- NODE 2: Tool Node ---
def tool_node(state: AgentState):
    last_message = state["messages"][-1]
    results = []
    for tool_call in last_message.tool_calls:
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]
        tool_result = tool_executor[tool_name].invoke(tool_args)
        results.append(ToolMessage(
            content=tool_result,
            tool_call_id=tool_call["id"]
        ))
    return {"messages": results}

# --- ROUTER ---
def should_continue(state: AgentState):
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "tool_node"
    return END

# InMemorySaver — same concept as SqliteSaver but stored in RAM
# Works with current LangGraph version, no context manager needed
from langgraph.checkpoint.memory import MemorySaver

checkpointer = MemorySaver()

graph_builder = StateGraph(AgentState)
graph_builder.add_node("agent_node", agent_node)
graph_builder.add_node("tool_node", tool_node)
graph_builder.set_entry_point("agent_node")
graph_builder.add_conditional_edges("agent_node", should_continue, {
    "tool_node": "tool_node",
    END: END
})
graph_builder.add_edge("tool_node", "agent_node")

# Compile graph WITH checkpointer
graph = graph_builder.compile(checkpointer=checkpointer)

print("Graph compiled with MemorySaver checkpointer")
print("Short term memory: enabled")

## Step 4: Test Short Term Memory — Multi Turn Conversation
The key test for short term memory is a follow up message.
Customer says "my order is late" — then follows up with "I want a refund".
Without memory the agent would ask "which order?" every time.
With memory the agent remembers the order ID from the previous turn.
We use thread_id to identify the conversation session.

In [ ]:
# thread_id is like a session ID — it links messages in the same conversation
# Same thread_id = same conversation = agent remembers previous messages
# Different thread_id = new conversation = agent starts fresh

SYSTEM_MESSAGE = SystemMessage(content="""You are a helpful Swiggy customer
support agent. When a customer has a complaint, first check the order status,
then take action. Only give a final answer when you have all the information
you need.""")

def chat(message, thread_id="thread_1"):
    """
    Send one message to the agent in a conversation thread.
    thread_id links messages together — same id = same conversation.
    """
    config = {"configurable": {"thread_id": thread_id}}  # identifies the session

    result = graph.invoke(
        {"messages": [SYSTEM_MESSAGE, HumanMessage(content=message)]},
        config=config    # pass thread_id so LangGraph knows which session this is
    )

    # Return just the final answer
    return result["messages"][-1].content

# --- TEST: Multi turn conversation ---
# Turn 1: Customer asks about order
print("Turn 1:")
print("Customer: My order ORD001 is very late")
response1 = chat("My order ORD001 is very late", thread_id="customer_rahul")
print(f"Agent: {response1}")
print()

# Turn 2: Follow up — no order ID mentioned this time
# Agent should remember ORD001 from Turn 1
print("Turn 2:")
print("Customer: I want a refund for that")
response2 = chat("I want a refund for that", thread_id="customer_rahul")
print(f"Agent: {response2}")

## Step 5: Long Term Memory with Mem0
Long term memory = remembering customer across sessions.
Short term memory resets when the conversation ends.
Long term memory persists forever — across days, weeks, sessions.
Mem0 works in two steps:
1. SAVE — after each conversation, save key facts about the customer
2. RETRIEVE — before responding, fetch relevant memories about this customer
We identify each customer with a user_id — like a customer ID in Swiggy.

In [ ]:
def save_to_long_term_memory(user_id, conversation_summary):
    """
    Save important facts about a customer to Mem0.
    """
    mem0.add(
        conversation_summary,
        user_id=user_id
    )
    print(f"Saved to long term memory for user: {user_id}")

def get_long_term_memory(user_id, query):
    """
    Retrieve relevant memories about a customer from Mem0.
    """
    memories = mem0.search(
        query,
        filters={"user_id": user_id}
    )

    # Debug: let us see exactly what Mem0 returns
    print("Raw Mem0 response type:", type(memories))
    print("Raw Mem0 response:", memories)
    print()

    # Handle both possible response formats
    # Format 1: list of dicts — memories is [{"memory": "...", ...}]
    # Format 2: dict with results key — memories is {"results": [...]}
    if isinstance(memories, dict):
        results = memories.get("results", [])
    else:
        results = memories

    if results:
        memory_text = "\n".join([
            f"- {m['memory']}" if isinstance(m, dict) else f"- {m}"
            for m in results
        ])
        return f"Customer history:\n{memory_text}"
    else:
        return "No previous history found for this customer."

# Test retrieve
memories = get_long_term_memory(
    user_id="CUST001",
    query="complaints and refunds"
)
print("Retrieved memories:")
print(memories)

## Step 6: Combine Both Memories — Full Agent
Now we combine short term memory (LangGraph MemorySaver)
and long term memory (Mem0) into one agent function.
Before responding — fetch customer history from Mem0
After responding — save new facts to Mem0
During conversation — LangGraph remembers all turns automatically

In [ ]:
def run_agent_with_memory(user_message, user_id, thread_id):
    """
    Full agent with both memory types:
    1. Fetch long term memory from Mem0 before responding
    2. LangGraph MemorySaver handles short term memory automatically
    3. Save new facts to Mem0 after responding
    """
    print(f"Customer ({user_id}): {user_message}")
    print("-" * 50)

    # --- STEP 1: Fetch long term memory ---
    long_term = get_long_term_memory(user_id, user_message)
    print(f"Long term memory: {long_term}")
    print()

    # --- STEP 2: Build system message with customer history injected ---
    system_message = SystemMessage(content=f"""You are a helpful Swiggy customer
support agent. When a customer has a complaint, first check the order status,
then take action. Only give a final answer when you have all the information
you need.

{long_term}

Use this history to personalise your response. If customer had a previous
refund, acknowledge it. If they are a Swiggy One member, mention their benefits.""")

    # --- STEP 3: Run agent with short term memory ---
    config = {"configurable": {"thread_id": thread_id}}

    result = graph.invoke(
        {"messages": [system_message, HumanMessage(content=user_message)]},
        config=config
    )

    final_answer = result["messages"][-1].content

    # --- STEP 4: Save this interaction to long term memory ---
    save_to_long_term_memory(
        user_id=user_id,
        conversation_summary=f"Customer said: {user_message}. Agent responded: {final_answer}"
    )

    print(f"Agent: {final_answer}")
    print()
    return final_answer

# --- TEST: New session, same customer ---
# This simulates Rahul calling back next day
# Agent should remember his previous refund from long term memory
print("=" * 60)
print("NEW SESSION — Rahul calls back next day")
print("=" * 60)

run_agent_with_memory(
    user_message="Hi, I had an issue with my order last time. Can you help me again?",
    user_id="CUST001",
    thread_id="session_day2"    # new thread_id = new session
)

## Summary

### What we built:
A Swiggy support agent with two real industry memory systems:
- Short term: LangGraph MemorySaver — remembers current conversation
- Long term: Mem0 — remembers customer across sessions

### How they work together:
1. Customer sends message
2. Agent fetches long term memory from Mem0 (who is this customer?)
3. Agent injects history into system prompt for personalisation
4. LangGraph MemorySaver handles short term memory automatically
5. After responding — new facts saved back to Mem0

### Key concepts:
- thread_id — identifies a conversation session (short term)
- user_id — identifies a customer across all sessions (long term)
- Short term resets when session ends
- Long term persists forever across days and sessions
- Mem0 auto-extracts key facts — we don't need to manually tag memories

### Short term vs Long term:
| | Short Term | Long Term |
|---|---|---|
| Tool | LangGraph MemorySaver | Mem0 |
| Scope | One conversation | Across all conversations |
| Resets | When session ends | Never |
| Identifies by | thread_id | user_id |
| Real world | RAM / Redis session | DynamoDB / PostgreSQL |

### AWS Equivalent:
| This session | AWS |
|---|---|
| LangGraph MemorySaver | Amazon ElastiCache (Redis) |
| Mem0 long term memory | Amazon DynamoDB + Bedrock Knowledge Bases |
| thread_id | AWS session token |
| user_id | Amazon Cognito user ID |
| Full agent with memory | Amazon Bedrock Agents with memory enabled |

### Next session:
Module 5 Session 5 — Multi Agent Systems
We wire up multiple specialist agents — Order agent, Refund agent,
Router agent — that hand work to each other automatically.